In [26]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, length, initcap, count
from pyspark.sql import Row

spark = SparkSession.builder.appName("PySpark Ass2").getOrCreate()


In [27]:
# 1.) Combine two DataFrames
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("CombineDataFrames").getOrCreate()
data1 = [("apple", 3, 5),
         ("banana", 1, 10),
         ("orange", 2, 8)]
df1 = spark.createDataFrame(data1, ["Name", "Col_1", "Col_2"])
data2 = [("apple", 3, 5),
         ("banana", 1, 15),
         ("grape", 4, 6)]
df2 = spark.createDataFrame(data2, ["Name", "Col_1", "Col_3"])
df1 = df1.select("Name", "Col_1", "Col_2")     
df2 = df2.withColumnRenamed("Col_2", "Col_3") 
df_final = df1.union(df2)
df_final.show()

+------+-----+-----+
|  Name|Col_1|Col_2|
+------+-----+-----+
| apple|    3|    5|
|banana|    1|   10|
|orange|    2|    8|
| apple|    3|    5|
|banana|    1|   15|
| grape|    4|    6|
+------+-----+-----+



In [28]:
# 2.) Extract only Email IDs
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
spark = SparkSession.builder.appName("ExtractEmails").getOrCreate()
data = [
    ("buying books at amazom.com",),
    ("rameses@egypt.com",),
    ("matt@t.co",),
    ("narendra@modi.com",)
]
df = spark.createDataFrame(data, ["value"])
df_emails = df.filter(col("value").like("%@%"))
df_emails.show(truncate=False)

+-----------------+
|value            |
+-----------------+
|rameses@egypt.com|
|matt@t.co        |
|narendra@modi.com|
+-----------------+



In [29]:
# 3.) We want to show this : Pivot EU & US Revenue by Quarter
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
spark = SparkSession.builder.appName("PivotRevenue").getOrCreate()
data = [
    (2021, 1, "US", 5000),
    (2021, 1, "EU", 4000),
    (2021, 2, "US", 5500),
    (2021, 2, "EU", 4500),
    (2021, 3, "US", 6000),
    (2021, 3, "EU", 5000),
    (2021, 4, "US", 7000),
    (2021, 4, "EU", 6000)
]
df = spark.createDataFrame(data, ["year", "quarter", "region", "revenue"])
df_pivot = df.groupBy("year", "quarter").pivot("region").sum("revenue")
df_pivot.orderBy("quarter").show()

+----+-------+----+----+
|year|quarter|  EU|  US|
+----+-------+----+----+
|2021|      1|4000|5000|
|2021|      2|4500|5500|
|2021|      3|5000|6000|
|2021|      4|6000|7000|
+----+-------+----+----+



In [30]:
#4.) Replace Missing Spaces in a String with the Least Frequent Character
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, udf
from pyspark.sql.types import StringType
from collections import Counter

spark = SparkSession.builder.appName("ReplaceSpaces").getOrCreate()
data = [("dbc deb abed gade",)]
df = spark.createDataFrame(data, ["string"])
def replace_spaces_machaa(s):
    freq = Counter(s.replace(" ", ""))
    least_char = min(freq, key=freq.get)  
    return s.replace(" ", least_char)

replace_udf = udf(replace_spaces_machaa, StringType())
df_final = df.withColumn("Updated_string", replace_udf(col("string")))
df_final.show(truncate=False)

+-----------------+-----------------+
|string           |Updated_string   |
+-----------------+-----------------+
|dbc deb abed gade|dbccdebcabedcgade|
+-----------------+-----------------+



In [31]:
#5.) Check if a DataFrame Has Missing Values & Count Them in Each Column
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder.appName("MissingValues").getOrCreate()
data = [
    ("A", 1, None),
    ("B", None, 123),
    ("B", 3, 456),
    ("D", None, None)
]
df = spark.createDataFrame(data, ["Name", "Value", "id"])
count = {}
for column in df.columns:
    count_nulls = df.filter(col(column).isNull()).count()
    count[column] = count_nulls
    
has_nulls = any(count > 0 for count in count.values())
print(has_nulls)
print(count)

True
{'Name': 0, 'Value': 2, 'id': 2}


In [32]:
#6.) Filter Every Nth Row in a DataFrame
from pyspark.sql import SparkSession
from pyspark.sql.functions import row_number, col
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("FilterEveryNthRow").getOrCreate()
data = [
    ("Alice", 1),
    ("Bob", 2),
    ("Charlie", 3),
    ("Dave", 4),
    ("Eve", 5),
    ("Frank", 6),
    ("Grace", 7),
    ("Hannah", 8),
    ("Igor", 9),
    ("Jack", 10)
]
df = spark.createDataFrame(data, ["Name", "Number"])
windowSpec = Window.orderBy("Number")
df = df.withColumn("rn", row_number().over(windowSpec))
n = 3
df_filtered = df.filter(col("rn") % n == 0)
df_filtered.show()

+-------+------+---+
|   Name|Number| rn|
+-------+------+---+
|Charlie|     3|  3|
|  Frank|     6|  6|
|   Igor|     9|  9|
+-------+------+---+



In [33]:
#7.) Just to check the Matching Columns
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when

spark = SparkSession.builder.appName("CommonColumns").getOrCreate()
data = [("John", "John"), ("Lily", "Lucy"), ("Sam", "Sam"), ("Lucy", "Lily")]
df = spark.createDataFrame(data, ["Name1", "Name2"])
df = df.withColumn("Match", when(col("Name1") == col("Name2"), True).otherwise(False))
df.show()

+-----+-----+-----+
|Name1|Name2|Match|
+-----+-----+-----+
| John| John| true|
| Lily| Lucy|false|
|  Sam|  Sam| true|
| Lucy| Lily|false|
+-----+-----+-----+



In [34]:
#8.) Create lag and lead of a column by group
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql.functions import lag, lead, col
spark = SparkSession.builder.appName("LagLeadByGroup").getOrCreate()
data = [
    ("2023-01-01", "Store1", 100),
    ("2023-01-02", "Store1", 150),
    ("2023-01-03", "Store1", 200),
    ("2023-01-04", "Store1", 250),
    ("2023-01-05", "Store1", 300),
    ("2023-01-01", "Store2", 50),
    ("2023-01-02", "Store2", 60),
    ("2023-01-03", "Store2", 80),
    ("2023-01-04", "Store2", 90),
    ("2023-01-05", "Store2", 120)
]
df = spark.createDataFrame(data, ["Date", "Store", "Sales"])
windowSpec = Window.partitionBy("Store").orderBy("Date")
df = df.withColumn("Lag_Sales", lag("Sales").over(windowSpec))
df = df.withColumn("Lead_Sales", lead("Sales").over(windowSpec))
df.show()

+----------+------+-----+---------+----------+
|      Date| Store|Sales|Lag_Sales|Lead_Sales|
+----------+------+-----+---------+----------+
|2023-01-01|Store1|  100|     NULL|       150|
|2023-01-02|Store1|  150|      100|       200|
|2023-01-03|Store1|  200|      150|       250|
|2023-01-04|Store1|  250|      200|       300|
|2023-01-05|Store1|  300|      250|      NULL|
|2023-01-01|Store2|   50|     NULL|        60|
|2023-01-02|Store2|   60|       50|        80|
|2023-01-03|Store2|   80|       60|        90|
|2023-01-04|Store2|   90|       80|       120|
|2023-01-05|Store2|  120|       90|      NULL|
+----------+------+-----+---------+----------+



In [35]:
#9.) Get frequency of unique values in the entire dataframe
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, explode, array, count
spark = SparkSession.builder.appName("ValueFrequency").getOrCreate()
data = [
    (1, 2, 3),
    (2, 3, 4),
    (1, 2, 3),
    (4, 5, 6),
    (2, 3, 4)
]
df = spark.createDataFrame(data, ["Column1", "Column2", "Column3"])
df_single = df.select(explode(array(*df.columns)).alias("Total_Values"))
df_count = df_single.groupBy("Total_Values").agg(count("*").alias("count")).orderBy(col("count").desc())
df_count.show()

+------------+-----+
|Total_Values|count|
+------------+-----+
|           3|    4|
|           2|    4|
|           4|    3|
|           1|    2|
|           6|    1|
|           5|    1|
+------------+-----+



In [36]:
#10.) Reverse the rows of a datafram
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col
spark = SparkSession.builder.appName("ReverseRows").getOrCreate()
data = [
    (1, 2, 3, 4),
    (2, 3, 4, 5),
    (3, 4, 5, 6),
    (4, 5, 6, 7)
]
df = spark.createDataFrame(data, ["col_1", "col_2", "col_3", "col_4"])
windowSpec = Window.orderBy(col("col_1"))
df = df.withColumn("rn", row_number().over(windowSpec))
df_reversed = df.orderBy(col("rn").desc()).drop("rn")
df_reversed.show()

+-----+-----+-----+-----+
|col_1|col_2|col_3|col_4|
+-----+-----+-----+-----+
|    4|    5|    6|    7|
|    3|    4|    5|    6|
|    2|    3|    4|    5|
|    1|    2|    3|    4|
+-----+-----+-----+-----+

